# Import libraries

In [1]:
# Standard code libraries
import faulthandler
import json
import re
from datetime import datetime as dt
from io import StringIO
from pathlib import Path
from typing import TYPE_CHECKING, cast

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# Custom code libraries from ReSurfEMG
from resurfemg.data_connector.config import ConfigCreatorWidget
from resurfemg.data_connector.file_discovery import find_files
from resurfemg.data_connector.tmsisdk_lite import Poly5Reader
from resurfemg.pipelines.ipy_widgets import DatasetSelector as ds  # noqa: N813
from resurfemg.pipelines.multimodal import MultimodalDataGroup

if TYPE_CHECKING:
    from resurfemg.data_connector.data_classes import (
        EmgDataGroup,
        VentilatorDataGroup,
    )

# from resurfemg.postprocessing import quality_assessment as qa

faulthandler.enable()

# %matplotlib widget
%matplotlib qt

from IPython.display import HTML


def show(df, n_rows=None, n_cols=None):
    with pd.option_context('display.max_rows', n_rows, 'display.max_columns', n_cols):  # more options can be specified also
        display(HTML(df._repr_html_()))

# Discover files

In [ ]:
creator = ConfigCreatorWidget("config_IC.json")

In [ ]:
# Identify all recordings available
# for the selected patient/measurement_date

# Root directory for test data
config = creator.get_config()
root_patient_data_directory = config.get_directory("patient_data")


files = find_files(root_patient_data_directory)
# # _patient_regex = r"^([Pp]_?\d+)"  # note the ( ) around the patternprint(files)
_patient_regex = r"MST\d{3}"
# col = files["files"]

# ids = col.str.extract(_patient_regex, expand=False)
# dataset_dict = col.groupby(ids).apply(list).to_dict()

ps = ds(root_directory=root_patient_data_directory, patient_regex=_patient_regex)

In [ ]:
# import the protocol details

PS_PEEP_levels_df = pd.read_csv(
    "C:\\Users\\ManinettiC\\Documents\\Participant_Id measurement_date PS_level.csv",
    delimiter="\t",
)

PS_PEEP_levels_df.columns = PS_PEEP_levels_df.columns.str.replace(
    "PS_level_(cmH2O)_Step_", "PS_cmH2O_"
).str.replace("PS_Recording_nr_Step_", "PS_Recording_nr_")
PS_PEEP_levels_df.columns = PS_PEEP_levels_df.columns.str.replace(
    "PEEP_protocol_PEEP_level_(cmH2O)_Step_", "PEEP_cmH2O_"
).str.replace("PEEP_protocol_Recording_nr_Step_", "PEEP_Recording_nr_")

PS_dict = {}
levels = {
    "PS": ["minus_3", "0", "plus_3", "plus_6"],
    "PEEP": ["minus_2", "0", "plus_2", "plus_4"],
}

for row in PS_PEEP_levels_df.itertuples(index=False):
    patient_id = row.Participant_Id
    measurement_date = row.measurement_date
    for _type, _levels in levels.items():
        for level in _levels:
            if getattr(row, f"{_type}_cmH2O_{level}") is not None:
                _recording_nr = f"{getattr(row, f'{_type}_Recording_nr_{level}'):03d}"
                if patient_id not in PS_dict:
                    PS_dict[patient_id] = {}
                if measurement_date not in PS_dict[patient_id]:
                    PS_dict[patient_id][measurement_date] = {}
                if _recording_nr not in PS_dict[patient_id][measurement_date]:
                    PS_dict[patient_id][measurement_date][_recording_nr] = {}
                PS_dict[patient_id][measurement_date][_recording_nr]["cmH2O"] = getattr(
                    row, f"{_type}_cmH2O_{level}"
                )
                PS_dict[patient_id][measurement_date][_recording_nr]["level"] = level
                PS_dict[patient_id][measurement_date][_recording_nr]["protocol"] = _type


In [ ]:
# Add the filenames
# to the protocol details dictionary
dataset_dict = ps.dataset_dict
records = pd.DataFrame(columns=["patient", "day", "step", "PEEP"])
format_day = r"^\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}$"
format_step = "([0-9]{3})"
for patient, record in dataset_dict.items():
    for _day, day_record in record.items():
        if not re.match(format_day, _day):
            print(
                f"Discarding '{_day}' for patient '{patient}'"
                f"as it does not match the expected format."
            )
            continue
        day = dt.strptime(_day, "%Y-%m-%d_%H-%M-%S").strftime("%d/%m/%Y")  # noqa: DTZ007
        for step, step_record in day_record.items():
            if not re.match(format_step, step):
                print(
                    f"Discarding '{step}' for patient '{patient}' on day '{day}'"
                    f"as it does not match the expected format."
                )
                continue
            # print(patient, day, step, record)
            try:
                for file in step_record:
                    _type = ps._guess_data_type(file)  # noqa: SLF001
                    _file = Path(ps.root_directory).joinpath(patient, _day, step, file)
                    PS_dict[patient][day][step][_type] = _file
            except Exception as e:  # noqa: BLE001
                print(f"Error occurred while processing file '{file}': {e}")  # type: ignore
                continue


In [109]:
# export the protocol details
# dictionary to a JSON file

with Path.open(Path("C:\\Users\\ManinettiC\\Documents\\PS_dict.json"), "w") as f:
    json.dump(PS_dict, f, indent=4, default=str)

# Process files (auto)

In [ ]:
# try:
# from copy import deepcopy

PS_dict = json.load(Path.open(Path("C:\\Users\\ManinettiC\\Documents\\PS_dict.json")))
for patient, patient_record in PS_dict.items():
    for day, day_record in patient_record.items():
        for step, record in day_record.items():
            if "Ventilator" in record and "EMG" in record:
                protocol = record["protocol"]
                record_string = (
                    f"subject {patient}, protocol {protocol}, day {day}, "
                    f"step {step} ({record['cmH2O']} cmH2O)"
                )
                print(f"\033[94m Processing {record_string}")
                _d_temp = record.copy()
                dump_file = Path.joinpath(
                    Path("C:\\Users\\ManinettiC\\Documents\\TOPSPIN"),
                    patient,
                    protocol,
                    str.replace(day, "/", "_"),
                    f"{record['cmH2O']}cmH2O.json",
                )
                dump_file.parent.mkdir(parents=True, exist_ok=True)
                if dump_file.exists():
                    print("File already exists, skipping.")
                    continue
                try:
                    multimodal_data_group = MultimodalDataGroup(
                        cast(
                            "VentilatorDataGroup",
                            ds.get_data(
                                Poly5Reader(str(record["Ventilator"]), verbose=False),
                                "Ventilator",
                                verbose=False,
                            ),
                        ),
                        cast(
                            "EmgDataGroup",
                            ds.get_data(
                                Poly5Reader(str(record["EMG"]), verbose=False),
                                "EMG",
                                verbose=False,
                            ),
                        ),
                    )
                    multimodal_data_group.set_labels()
                    multimodal_data_group.process_emg()
                    multimodal_data_group.process_emg_breaths(trial=protocol)
                    if protocol == "PEEP":
                        try:
                            multimodal_data_group.process_ventilator_breaths(
                                trial=protocol
                            )
                            multimodal_data_group.link_breaths(trial=protocol)
                            multimodal_data_group.calculate_ETP(trial=protocol)
                            multimodal_data_group.calculate_PTP()
                            multimodal_data_group.compute_quality_criteria(trial=protocol)
                            peak_set_name = "Pocc"
                            vent_idx = multimodal_data_group.p_vent_idx
                        except Exception as e:
                            print(f"\033[93m >>>>> Error at {record_string}: {e}")
                            continue
                    else:
                        try:
                            multimodal_data_group.calculate_ETP(trial=protocol)
                            multimodal_data_group.compute_quality_criteria(trial=protocol)
                            multimodal_data_group.process_ventilator_breaths(
                                trial=protocol
                            )
                            multimodal_data_group.link_breaths(trial=protocol)
                            peak_set_name = "neural_breaths"
                            vent_idx = multimodal_data_group.v_vent_idx
                        except Exception as e:
                            print(f"\033[93m >>>>> Error at {record_string}: {e}")
                            continue

                    for df_name in [
                        "peak_df",
                        "quality_values_df",
                        "quality_outcomes_df",
                    ]:
                        if "vent" not in _d_temp:
                            _d_temp["vent"] = {}
                        _d_temp["vent"][df_name] = getattr(
                            multimodal_data_group.vent_timeseries[vent_idx].peaks[
                                peak_set_name
                            ],
                            df_name,
                        )
                        for idx in multimodal_data_group.emg_idx:
                            label = multimodal_data_group.emg_timeseries.labels[idx]
                            if label not in _d_temp:
                                _d_temp[label] = {}
                            _d_temp[label][df_name] = getattr(
                                multimodal_data_group.emg_timeseries[label].peaks[
                                    peak_set_name
                                ],
                                df_name,
                            )

                    with Path.open(Path(dump_file), "w") as f:
                        json.dump(_d_temp, f, indent=4, default=lambda df: df.to_json())
                    print(f"\033[92m >>>>> Successfully processed {record_string}")
                except Exception as e:
                    print(f"\033[93m >>>>> Error at {record_string}: {e}")
                    continue
# except Exception as e:
#     print(f">>>>> Error at {record_string}: {e}")

# Import results

For each protocol (PS, PEEP) and each measure (ventilator, EMGdi, EMGic), generate three dataframes: peak_df, quality_values_df, quality_outcomes_df. 
Each protocol's own dataframes, including the information about the patient, measurement day, and protocol settings, are iteratively appended to the corresponding comprehensive dataframe.

In [2]:
# load the trials dictionary
PS_dict = json.load(Path.open(Path("C:\\Users\\ManinettiC\\Documents\\PS_dict.json")))


In [40]:
measure_names = {
    "PS": ["vent", "EMGdi", "EMGic"],
    "PEEP": ["vent", "EMGdi", "EMGic"],
}

df_names = ["peak_df", "quality_values_df", "quality_outcomes_df"]
results_dict = {"PS": {}, "PEEP": {}}

for protocol_name, protocol_df_names in measure_names.items():
    for measure in protocol_df_names:
        results_dict[protocol_name][measure] = {}
        for df_name in df_names:
            results_dict[protocol_name][measure][df_name] = pd.DataFrame()

_trial_info = ["patient", "day", "step", "level"]
# _index =[[["info"]*len(_trial_info), ["percentage_valid_breaths"]*3], list(_trial_info) + measure_names["PS"]]
# _index = pd.MultiIndex.from_arrays(
#     [
#         [""] * len(_trial_info) + ["percentage_valid_breaths"] * len(measure_names["PS"]),
#         list(_trial_info) + measure_names["PS"],
#     ]
# )
cols_ps = [
    f"{prefix}{measure}"
    for prefix in ("valid_percentage_", "valid_breaths_", "tot_breaths_")
    for measure in measure_names["PS"]
]
trial_details = {"PS": pd.DataFrame(columns=_trial_info + cols_ps), "PEEP": pd.DataFrame(columns=_trial_info + ["valid_manoeuvers_" + meas for meas in measure_names['PEEP']])}

PS_dict = json.load(Path.open(Path("C:\\Users\\ManinettiC\\Documents\\PS_dict.json")))
for patient, patient_record in PS_dict.items():
    for day, day_record in patient_record.items():
        for step, record in day_record.items():
            if "Ventilator" in record and "EMG" in record:
                protocol = record["trial"]
                level = record["cmH2O"]
                record_string = (
                    f"patient {patient}, protocol {protocol}, day {day}, "
                    f"step {step} ({level} cmH2O)"
                )
                print(f"\033[94m Processing {record_string}")

                dump_file = Path.joinpath(
                    Path("C:\\Users\\ManinettiC\\Documents\\TOPSPIN_mixed_baseline"),
                    patient,
                    protocol,
                    str.replace(day, "/", "_"),
                    f"{level}cmH2O.json",
                )
                if not dump_file.exists():
                    print(f"\033[91m >>>>> File {dump_file} does not exist, skipping.")
                    continue

                _d_temp = json.load(Path.open(Path(dump_file)))
                valid_percentage = {}
                valid_breaths = {}
                tot_breaths = {}
                for measure in measure_names[protocol]:
                    valid_percentage[measure] = None
                    for df_name in df_names:
                        _df_temp = pd.DataFrame()
                        if df_name not in _d_temp[measure]:
                            print(
                                f"\033[91m >>>>> DataFrame '{df_name}' not found in {dump_file}, skipping."
                            )
                            continue
                        _df_temp = pd.read_json(StringIO(_d_temp[measure][df_name]))
                        _df_temp["patient"] = patient
                        _df_temp["day"] = day
                        _df_temp["level"] = level
                        _df_temp["step"] = step
                        _d_temp[df_name] = _df_temp
                        results_dict[protocol][measure][df_name] = pd.concat(
                            [results_dict[protocol][measure][df_name], _df_temp],
                            ignore_index=True,
                        )
                        if df_name == "peak_df":
                            valid = _df_temp['valid']
                            tot_breaths[measure] = valid.count()
                            valid_breaths[measure] = valid[valid == 1].count()

                i = len(trial_details[protocol])

                if protocol == "PS":

                    for measure in measure_names["PS"]:
                        trial_details[protocol].loc[i, "tot_breaths_" + measure] = tot_breaths.get(measure, None)
                        trial_details[protocol].loc[i, "valid_breaths_" + measure] = valid_breaths.get(measure, None)
                        trial_details[protocol].loc[i, "valid_percentage_" + measure] = valid_breaths.get(measure, None)/tot_breaths.get('vent', 0) * 100
                else:
                    for measure in measure_names["PEEP"]:
                        trial_details[protocol].loc[i, "valid_manoeuvers_" + measure] = valid_breaths.get(measure, None)
                    trial_details[protocol].loc[i, "manoeuvers"] = sum([_tot for _tot in tot_breaths.values()])/len(tot_breaths.keys())
                for var_name, var in zip(_trial_info, [patient, day, step, level]):
                    trial_details[protocol].loc[i, var_name] = var

 Processing patient MST001, protocol PS, day 24/12/2021, step 005 (12 cmH2O)
 Processing patient MST001, protocol PS, day 24/12/2021, step 004 (15 cmH2O)
 Processing patient MST001, protocol PS, day 24/12/2021, step 003 (18 cmH2O)
 Processing patient MST001, protocol PS, day 24/12/2021, step 002 (21 cmH2O)
 Processing patient MST001, protocol PEEP, day 24/12/2021, step 006 (3 cmH2O)
 Processing patient MST001, protocol PEEP, day 24/12/2021, step 007 (5 cmH2O)
 Processing patient MST001, protocol PEEP, day 24/12/2021, step 008 (7 cmH2O)
 Processing patient MST001, protocol PEEP, day 24/12/2021, step 009 (9 cmH2O)
 Processing patient MST001, protocol PS, day 03/01/2022, step 002 (3 cmH2O)
 Processing patient MST001, protocol PS, day 03/01/2022, step 003 (6 cmH2O)
 Processing patient MST001, protocol PS, day 03/01/2022, step 004 (9 cmH2O)
 Processing patient MST001, protocol PS, day 03/01/2022, step 005 (12 cmH2O)
 Processing patient MST001, protocol PEEP, day 03/01/2022, step 006 (3 cmH2

In [4]:
_index_ps = pd.MultiIndex.from_frame(trial_details['PS'][["patient", "day", "level"]], names=["patient", "day", "PS_level"])
ps_details= pd.DataFrame(trial_details['PS'].iloc[:, 4:])
ps_details.set_index(_index_ps, inplace=True)
# ps_details
# with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
#     print(ps_details)

_index_valid_ps = _index_ps.droplevel("PS_level").drop_duplicates()
_valid_cols = [col for col in ps_details.columns if "percentage" in col]
ps_details_valid = pd.DataFrame(columns=[col.replace("_percentage", "") for col in _valid_cols], index=_index_valid_ps)
for idx in _index_valid_ps:
    ps_details_valid.loc[idx][:] = (ps_details.loc[idx][_valid_cols] > 10).all()

print(ps_details_valid)

print("total valid trials: ", ps_details_valid.sum())

                   valid_vent valid_EMGdi valid_EMGic
patient day                                          
MST001  24/12/2021       True        True        True
        03/01/2022       True       False       False
        31/12/2021       True        True       False
        29/12/2021       True       False       False
        27/12/2021       True       False        True
MST002  07/01/2022       True       False       False
        10/01/2022       True       False       False
MST003  28/01/2022       True       False        True
        02/02/2022       True       False       False
        31/01/2022       True       False       False
        26/01/2022       True       False       False
MST004  02/02/2022       True        True       False
        07/02/2022       True       False       False
        28/01/2022       True        True       False
        04/02/2022       True        True       False
MST005  28/02/2022       True       False       False
MST006  18/03/2022       Tru

C:\Users\ManinettiC\AppData\Local\Temp\ipykernel_34080\2030937940.py:12: PerformanceWarning: indexing past lexsort depth may impact performance.
  ps_details_valid.loc[idx][:] = (ps_details.loc[idx][_valid_cols] > 10).all()
C:\Users\ManinettiC\AppData\Local\Temp\ipykernel_34080\2030937940.py:12: PerformanceWarning: indexing past lexsort depth may impact performance.
  ps_details_valid.loc[idx][:] = (ps_details.loc[idx][_valid_cols] > 10).all()
C:\Users\ManinettiC\AppData\Local\Temp\ipykernel_34080\2030937940.py:12: PerformanceWarning: indexing past lexsort depth may impact performance.
  ps_details_valid.loc[idx][:] = (ps_details.loc[idx][_valid_cols] > 10).all()
C:\Users\ManinettiC\AppData\Local\Temp\ipykernel_34080\2030937940.py:12: PerformanceWarning: indexing past lexsort depth may impact performance.
  ps_details_valid.loc[idx][:] = (ps_details.loc[idx][_valid_cols] > 10).all()
C:\Users\ManinettiC\AppData\Local\Temp\ipykernel_34080\2030937940.py:12: PerformanceWarning: indexing pa

In [148]:
results_dict['PS']['EMGdi']['peak_df']

,peak_idx,start_idx,end_idx,valid,AUB,aub_y_ref,ETP,bell_y_min,bell_a,bell_b,bell_c,sEA,patient,day,level,step
0,1075,909,1350,False,0.212203,1.212961,0.249137,1.212961,1.205787,0.556852,0.303833,0.256865,MST001,24/12/2021,12,005
1,2596,1350,4935,False,7.412600,6.445633,8.395258,1.212961,0.486906,0.767578,2.705439,-0.245310,MST001,24/12/2021,12,005
2,4048,1350,4935,False,7.412600,6.445633,8.395258,1.212961,0.436872,1.476562,4814.473082,-0.328502,MST001,24/12/2021,12,005
3,5827,4935,6499,True,0.762166,1.212961,2.512150,1.212961,4.814443,2.834592,0.327131,4.244217,MST001,24/12/2021,12,005
4,9015,6499,10395,False,7.967226,6.445633,9.250651,1.212961,0.370087,3.901855,4724.773405,-0.569783,MST001,24/12/2021,12,005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61449,976025,975983,976040,False,0.062507,2.023625,0.064896,2.023625,2.347708,476.569906,0.096712,0.129510,MST017,09/12/2022,12,002
61450,978969,976040,980005,False,10.773678,9.756706,12.906697,2.023625,1.446873,478.512207,1.816138,-0.025020,MST017,09/12/2022,12,002
61451,981194,980005,981995,False,2.073585,2.023625,3.973338,2.023625,4.834374,479.087389,0.682153,3.559810,MST017,09/12/2022,12,002
61452,985778,985741,986148,False,0.506153,2.106474,0.556941,2.106474,3.030947,481.227306,0.706770,0.543249,MST017,09/12/2022,12,002


In [5]:
_index_peep = pd.MultiIndex.from_frame(trial_details['PEEP'][["patient", "day", "level"]], names=["patient", "day", "PEEP_level"])
peep_details= pd.DataFrame(trial_details['PEEP'].iloc[:, 4:])
peep_details.set_index(_index_peep, inplace=True)
peep_details
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    print(peep_details)

                              valid_manoeuvers_vent valid_manoeuvers_EMGdi  \
patient day        PEEP_level                                                
MST001  24/12/2021 3                              4                      0   
                   5                              3                      0   
                   7                              3                      1   
                   9                              3                      2   
        03/01/2022 3                              1                      0   
                   5                              3                      3   
                   7                              2                      2   
                   9                              3                      2   
        31/12/2021 3                              0                      1   
        29/12/2021 3                              1                      0   
                   5                              2             

In [6]:
# save the results
# now save the results to two json files (one for PEEP, one for PS)

for protocol, results in results_dict.items():
    _file = Path(r"C:\\Users\\ManinettiC\\Documents") / f"{protocol}_raw_results.json"
    with Path.open(_file, "w") as f:
        json.dump(results, f, indent=4, default=lambda df: df.to_json())

In [ ]:
# create a dataframe with the details of each trial (one for each protocol)

# PS protocol: valid trials have at least 10% valid neural breaths at each PS level

In [7]:
print("Counts (valid/total)")

for protocol, dfs in results_dict.items():
    print(f"--- {protocol} ---")
    for measure, measure_dfs in dfs.items():
        print(f"--- {measure} ---")
        _total_df = measure_dfs["peak_df"]
        # print(f"{label}")
        valid = _total_df["valid"]
        breaths_tot = _total_df['start_idx'].count()
        breaths_valid = _total_df.loc[valid, 'start_idx'].count()
        breaths_percent = (breaths_valid/breaths_tot)*100
        _valid_df = (
            _total_df.loc[valid]
            .drop_duplicates(["patient", "day"])
            .groupby("patient")
            .size()
        )
        _total_df = (
                    _total_df
                    .drop_duplicates(["patient", "day"])
                    .groupby("patient")
                    .size()
                )
        trials_valid = _valid_df.sum()
        trials_tot = _total_df.sum()
        trials_percent = (trials_valid/trials_tot)*100
        patients_valid = _valid_df.count()
        patients_tot = _total_df.count()
        patients_percent = (patients_valid/patients_tot)*100

        print(f"\tTotal breaths: {breaths_valid}/{breaths_tot} ({breaths_percent: .1f}%)")
        print(f"\tTotal trials: {trials_valid}/{trials_tot} ({trials_percent: .1f}%)")
        print(f"\tTotal patients: {patients_valid}/{patients_tot} ({patients_percent: .1f}%)")



Counts (valid/total)
--- PS ---
--- vent ---
	Total breaths: 68829/69347 ( 99.3%)
	Total trials: 41/41 ( 100.0%)
	Total patients: 17/17 ( 100.0%)
--- EMGdi ---
	Total breaths: 3224/61454 ( 5.2%)
	Total trials: 31/41 ( 75.6%)
	Total patients: 13/17 ( 76.5%)
--- EMGic ---
	Total breaths: 4065/69347 ( 5.9%)
	Total trials: 26/41 ( 63.4%)
	Total patients: 9/17 ( 52.9%)
--- PEEP ---
--- vent ---
	Total breaths: 227/391 ( 58.1%)
	Total trials: 38/41 ( 92.7%)
	Total patients: 16/17 ( 94.1%)
--- EMGdi ---
	Total breaths: 50/391 ( 12.8%)
	Total trials: 12/41 ( 29.3%)
	Total patients: 6/17 ( 35.3%)
--- EMGic ---
	Total breaths: 60/391 ( 15.3%)
	Total trials: 10/41 ( 24.4%)
	Total patients: 4/17 ( 23.5%)


In [8]:
# show the count of valid protocols for each measure
quality_df = results_dict['PEEP']['EMGdi']["quality_outcomes_df"]
quality_tests = quality_df.dtypes == bool
quality_stats = pd.DataFrame(index=quality_tests.index[quality_tests], columns=["total protocols", "valid_protocols", "invalid_protocols", "patients"])
for test in quality_tests.index[quality_tests]:
    _df = (
        quality_df.loc[quality_df[test]]
        # .drop_duplicates(["patient", "day", "step"])
        .groupby("patient")
        .size()
    )
    quality_stats.loc[test, "total protocols"] = quality_df.shape[0]
    quality_stats.loc[test, "valid_protocols"] = _df.sum()
    quality_stats.loc[test, "invalid_protocols"] = quality_stats.loc[test, "total protocols"] - _df.sum()
    quality_stats.loc[test, "patients"] = _df.nunique()
    # print(f"\t{test}: {_df.sum()} valid protocols, {_df.nunique()} patients")

quality_stats

,total protocols,valid_protocols,invalid_protocols,patients
baseline_detection,391,255,136,9
interpeak_distance,391,364,27,12
snr,391,150,241,9
aub,391,129,262,13
bell,391,375,16,10
relative_aub,391,390,1,12
relative_etp,391,383,8,13


In [9]:
results_dict['PEEP']['vent']['peak_df'].groupby(["patient", "day"]).size()

patient  day       
MST001   03/01/2022    11
         24/12/2021    13
         27/12/2021    14
         29/12/2021     9
         31/12/2021     2
MST002   07/01/2022    10
         10/01/2022     8
MST003   02/02/2022    10
         26/01/2022    12
         28/01/2022    12
         31/01/2022     8
MST004   02/02/2022    12
         04/02/2022    11
         07/02/2022    12
         28/01/2022    12
MST005   28/02/2022    10
MST006   18/03/2022    12
MST007   11/03/2022     3
         14/03/2022     6
         18/03/2022     6
MST008   25/03/2022    17
MST009   28/03/2022     8
         30/03/2022     2
MST010   25/04/2022     1
         29/04/2022    11
MST011   30/05/2022    12
MST012   01/08/2022     6
         27/07/2022     9
         29/07/2022     6
MST013   21/09/2022     8
MST014   03/10/2022    10
MST015   24/10/2022    10
         28/10/2022     8
MST016   28/10/2022    11
MST017   02/12/2022    14
         05/12/2022     9
         07/12/2022     8
         09/12/202

In [143]:
_df

,peak_idx,start_idx,end_idx,valid,patient,day,level,step
0,327,281,450,True,MST001,24/12/2021,12,005
1,582,535,694,True,MST001,24/12/2021,12,005
2,810,764,900,True,MST001,24/12/2021,12,005
3,1029,980,1123,True,MST001,24/12/2021,12,005
4,1252,1203,1346,True,MST001,24/12/2021,12,005
...,...,...,...,...,...,...,...,...
69342,47664,47576,47919,True,MST017,09/12/2022,12,002
69343,47664,47576,47919,True,MST017,09/12/2022,12,002
69344,47848,47576,47919,False,MST017,09/12/2022,12,002
69345,48026,47962,48119,True,MST017,09/12/2022,12,002


In [112]:
results_dict[protocol][signal]['peak_df']

,peak_idx,start_idx,end_idx,valid,AUB,aub_y_ref,PTPocc,Pocc,patient,day,level,step
0,32522,32457,32543,True,0.130836,3.4209,6.151575,-16.854200,MST001,24/12/2021,3,006
1,32737,32694,32758,True,0.092564,3.4209,6.288096,-16.404100,MST001,24/12/2021,3,006
2,34970,34923,34994,True,0.088250,3.3708,7.216023,-17.804400,MST001,24/12/2021,3,006
3,36102,36057,36126,True,0.057750,3.3208,6.714154,-17.854500,MST001,24/12/2021,3,006
4,40586,40496,40613,True,0.142367,5.4214,7.816000,-18.554600,MST001,24/12/2021,5,007
...,...,...,...,...,...,...,...,...,...,...,...,...
386,34360,34286,34420,True,0.234500,3.6709,10.128503,-17.954500,MST017,09/12/2022,3,006
387,38584,38497,38622,True,0.241250,4.5711,9.278760,-16.304101,MST017,09/12/2022,3,006
388,37469,37374,37561,True,0.881827,6.6217,26.727014,-31.816417,MST017,09/12/2022,5,007
389,31905,31811,31972,True,0.680651,7.7219,12.924678,-21.763901,MST017,09/12/2022,7,008


In [41]:
patient = "MST001"
day = "24/12/2021"
protocol = "PEEP"
# level = 5
signals = ["vent", "EMGdi", "EMGic"]

baseline = 9 if protocol == "PEEP" else 12
print(step)
if protocol == "PS":
    columns = [var + "_" + signal for signal in ["EMGdi", "EMGic"] for var in ["ETP", "sEA"]] + ["TV"]
elif protocol == "PEEP":
    columns = ["PTPocc_vent"] + [var + "_" + signal for signal in ["EMGdi", "EMGic"] for var in ["ETP"]]

results = pd.DataFrame(columns = ["patient", "day", "level"] + columns)
results_normalized = pd.DataFrame(columns = ["patient", "day", "level"] + columns)

# _results = pd.DataFrame(columns = ["patient", "day", "level""])
for signal in signals:
    if protocol == "PS":
        measures = ["TV"] if signal == "vent" else ["ETP", "sEA"]
    elif protocol == "PEEP":
        measures = ["PTPocc"] if signal == "vent" else ["ETP"]
    _df = pd.DataFrame()
    _df = results_dict[protocol][signal]["peak_df"].copy()
    mean_bs = _df.query(f"patient == '{patient}' and day == '{day}' and level == {baseline}")
    _selected = _df.query(f"patient == '{patient}' and day == '{day}'")
    results["level"] = _selected['level']
    results_normalized["level"] = _selected['level']

    _selected = _selected[["valid"] + measures]
    _selected.columns = [col + "_" + signal for col in _selected.columns]
    
    for measure in measures:
        meas_col_name = measure + "_" + signal
        _mean_bs = mean_bs[mean_bs["valid"]][measure].mean()
        # _std_bs = _df[(_df["patient"] == patient) & (_df["day"] == day) & (_df["level"] == baseline)][[measure]].std()
        # _std_bs = _std_bs.replace(0, 1)  # replace 0 std with NaN to avoid division by zero
        _selected[meas_col_name + "_normalized"] = _selected[meas_col_name].apply(lambda x: x / _mean_bs)
        results_normalized[meas_col_name] = _selected[meas_col_name + "_normalized"]
        results[meas_col_name] = _selected[meas_col_name]

results["patient"] = patient
results["day"] = day
results_normalized["patient"] = patient
results_normalized["day"] = day



if protocol == "PEEP":
    for measure in ["EMGdi", "EMGic"]:
        results_normalized[f"NME_{measure}"] = results_normalized["PTPocc_vent"] / results_normalized[f"ETP_{measure}"]
        results[f"NME_{measure}"] = results["PTPocc_vent"] / results[f"ETP_{measure}"]

display(results)
display(results_normalized)

009


,patient,day,level,PTPocc_vent,ETP_EMGdi,ETP_EMGic,NME_EMGdi,NME_EMGic
0,MST001,24/12/2021,3,6.151575,1.152286,7.711526,5.338584,0.797712
1,MST001,24/12/2021,3,6.288096,1.763920,8.945347,3.564841,0.702946
2,MST001,24/12/2021,3,7.216023,1.532039,8.982698,4.710078,0.803325
3,MST001,24/12/2021,3,6.714154,1.183022,9.062670,5.675428,0.740858
4,MST001,24/12/2021,5,7.816000,1.799900,11.635692,4.342464,0.671726
5,MST001,24/12/2021,5,6.951271,1.828830,9.648006,3.800939,0.720488
6,MST001,24/12/2021,5,6.612928,1.355809,8.101484,4.877476,0.816261
7,MST001,24/12/2021,7,7.765417,1.986766,12.020378,3.908573,0.646021
8,MST001,24/12/2021,7,7.123509,2.049779,9.705572,3.475257,0.733961
9,MST001,24/12/2021,7,7.020986,1.557128,9.788053,4.508934,0.717302


,patient,day,level,PTPocc_vent,ETP_EMGdi,ETP_EMGic,NME_EMGdi,NME_EMGic
0,MST001,24/12/2021,3,0.853028,0.561911,0.674705,1.518084,1.264298
1,MST001,24/12/2021,3,0.871959,0.860174,0.782656,1.013701,1.114103
2,MST001,24/12/2021,3,1.000633,0.747097,0.785924,1.339361,1.273194
3,MST001,24/12/2021,3,0.931040,0.576899,0.792921,1.613869,1.174191
4,MST001,24/12/2021,5,1.083831,0.877719,1.018042,1.234827,1.064623
5,MST001,24/12/2021,5,0.963921,0.891827,0.844133,1.080838,1.141906
6,MST001,24/12/2021,5,0.917003,0.661159,0.708823,1.386963,1.293698
7,MST001,24/12/2021,7,1.076817,0.968844,1.051699,1.111445,1.023883
8,MST001,24/12/2021,7,0.987805,0.999573,0.849170,0.988227,1.163259
9,MST001,24/12/2021,7,0.973588,0.759332,0.856386,1.282164,1.136856


In [53]:
import importlib
import resurfemg
importlib.reload(resurfemg)
import resurfemg
importlib.reload(resurfemg.pipelines.multimodal)
from resurfemg.pipelines.multimodal import MultimodalDataGroup
level = 3
step = next(_step for _step, d in PS_dict[patient][day].items() if d['trial']==protocol and d['cmH2O'] == level)
plt.close("all")
record = PS_dict[patient][day][step]

mdg = MultimodalDataGroup(
    cast(
        "VentilatorDataGroup",
        ds.get_data(
            Poly5Reader(str(record["Ventilator"]), verbose=False),
            "Ventilator",
            verbose=False,
        ),
    ),
    cast(
        "EmgDataGroup",
        ds.get_data(
            Poly5Reader(str(record["EMG"]), verbose=False),
            "EMG",
            verbose=False,
        ),
    ),
)
mdg.set_labels()
mdg.process_emg()
mdg.get_emg_breaths(trial=protocol)
if protocol == "PEEP":
    mdg.process_ventilator_breaths(trial=protocol)
    mdg.link_breaths(trial=protocol)
    mdg.detect_emg_on_offsets(trial=protocol)
    mdg.calculate_ETP(trial=protocol)
    mdg.calculate_PTP()
    mdg.compute_quality_criteria(trial=protocol)
    peak_set_name, vent_idx = "Pocc", mdg.p_vent_idx
else:
    mdg.detect_emg_on_offsets(trial=protocol)
    mdg.calculate_ETP(trial=protocol)
    mdg.compute_quality_criteria(trial=protocol)
    mdg.process_ventilator_breaths(trial=protocol)
    mdg.link_breaths(trial=protocol)
    peak_set_name, vent_idx = "neural_breaths", mdg.v_vent_idx
mdg.calculate_peak_values(trial=protocol)               
for df_name in ["peak_df", "quality_values_df", "quality_outcomes_df"]:
    _d_temp.setdefault("vent", {})[df_name] = getattr(
        mdg.vent_timeseries[vent_idx].peaks[peak_set_name], df_name
    )
    for idx in mdg.emg_idx:
        label = mdg.emg_timeseries.labels[idx]
        _d_temp.setdefault(label, {})[df_name] = getattr(
            mdg.emg_timeseries[label].peaks[peak_set_name], df_name
        )
mdg.plot_data(trial=protocol)

mdg.emg_timeseries[1].peaks['Pocc'].peak_df

c:\Users\ManinettiC\Documents\ReSurfEMG\resurfemg\data_connector\data_classes.py:1987: UserWarning: Peak set 'Pocc' not found in channel 0.
  warnings.warn(msg, UserWarning)
c:\Users\ManinettiC\Documents\ReSurfEMG\resurfemg\data_connector\data_classes.py:1987: UserWarning: Peak set 'Pocc' not found in channel 1.
  warnings.warn(msg, UserWarning)
c:\Users\ManinettiC\Documents\ReSurfEMG\resurfemg\data_connector\data_classes.py:1987: UserWarning: Peak set 'Pocc' not found in channel 2.
  warnings.warn(msg, UserWarning)


,peak_idx,start_idx,end_idx,valid,AUB,aub_y_ref,ETP,bell_y_min,bell_a,bell_b,bell_c
0,665216,664339,666276,True,0.283981,1.263534,1.152286,1.263534,1.955611,324.819983,0.347342
1,669878,668863,670765,True,0.278977,1.263534,1.763920,1.263534,3.435873,327.009319,0.298087
2,715348,714681,716072,True,0.222961,1.190744,1.532039,1.190744,3.381310,349.283775,0.281615
3,738547,737749,739222,True,0.218390,1.262627,1.183022,1.262627,2.815060,360.609413,0.244773


In [52]:
pks = mdg.emg_timeseries[1].peaks['Pocc'].peak_df
display((pks['end_idx'] - pks['start_idx'])/2048)

0    0.945801
1    0.928711
2    0.679199
3    0.719238
dtype: float64

In [55]:
self = mdg
plot_markers = True
if plot_markers is True:
    if protocol == "PS":
        markers = ["neural_breaths", "neural_breaths", "neural_breaths"]
    else:
        markers = ["Pocc", "Pocc", "Pocc"]
nrows = 3
ncols = 2
axes = plt.subplots(
    nrows=nrows, ncols=ncols, sharex=True, figsize=(ncols * 6, nrows * 2)
)[1]

axes_emg = axes[:, 0]
axes_vent = axes[:, 1]

self.emg_timeseries.plot_full(
    axes=axes_emg,
    baseline_bool=True,
)

self.vent_timeseries.plot_full(
    axes=axes_vent,
    baseline_bool=True,
)

if plot_markers is True:
    self.emg_timeseries.plot_markers(
        peak_set_name=markers,
        axes=np.transpose([axes_emg]),
        colors=["tab:red", "tab:green", "tab:green"],
    )
    self.vent_timeseries.plot_markers(
        peak_set_name=markers,
        axes=np.transpose([axes_vent]),
        colors=["tab:red", "tab:green", "tab:green"],
    )

axes_emg[0].set_title("EMG data")
axes_emg[-1].set_xlabel("t (s)")
axes_vent[0].set_title("Ventilator data")
axes_vent[-1].set_xlabel("t (s)")
plt.show()

c:\Users\ManinettiC\Documents\ReSurfEMG\resurfemg\data_connector\data_classes.py:1987: UserWarning: Peak set 'neural_breaths' not found in channel 1.
  warnings.warn(msg, UserWarning)


In [ ]:
_sel = _df[(_df["patient"] == patient) & (_df["day"] == day) & (_df["level"] == level)]
_sel = _sel[_sel["valid"] == True]

_sel["sum"] = _sel["AUB"]+_sel["ETP"] + _sel["aub_y_ref"]
_sel

In [ ]:

_quality_results = {}

# Process file (manual)

In [ ]:
# ensure the correct labels are set

vent_timeseries.v_vent_idx = next(
    i for i, s in enumerate(vent_timeseries.labels) if "V" in s
)
vent_timeseries.p_vent_idx = next(
    i for i, s in enumerate(vent_timeseries.labels) if "P" in s
)
vent_timeseries.f_idx = next(
    i for i, s in enumerate(vent_timeseries.labels) if "F" in s
)

emg_timeseries.labels = ["ECG", "EMGdi", "EMGic"]
for idx, label in enumerate(emg_timeseries.labels):
    emg_timeseries[idx].label = label

In [ ]:
# plot the imported data

plt.close("all")

nrows = 3
ncols = 2
axes = plt.subplots(
    nrows=nrows, ncols=ncols, sharex=True, figsize=(ncols * 6, nrows * 2)
)[1]

axes_emg = axes[:, 0]
axes_vent = axes[:, 1]

emg_timeseries.plot_full(
    axes=axes_emg,
)


vent_timeseries.plot_full(
    axes=axes_vent,
)


axes_emg[0].set_title("EMG data")
axes_emg[-1].set_xlabel("t (s)")
axes_vent[0].set_title("Ventilator data")
axes_vent[-1].set_xlabel("t (s)")
plt.show()

# 3. Processing of ventilator signals

## PEEP settings and moving baseline

In [ ]:
# compute PEEP
vent_timeseries.find_peep()
# compute moving baseline
vent_timeseries.baseline(
    channel_idxs=[vent_timeseries.p_vent_idx], signal_io=("raw", "baseline")
)

## Pocc peak detection and PTPocc calculation

In [ ]:
# detect occlusion pressure peaks
vent_timeseries.find_occluded_breaths()

# detectPocc on- and offset
vent_timeseries[vent_timeseries.p_vent_idx].peaks["Pocc"].detect_on_offset(
    baseline=vent_timeseries[vent_timeseries.p_vent_idx]["baseline"],
    fs=vent_timeseries.fs,
)

# calculate PTPocc as the area between Paw and the moving baseline between the on- and offset of the Pocc peak
vent_timeseries[vent_timeseries.p_vent_idx].calculate_time_products(
    peak_set_name="Pocc",
    parameter_name="PTPocc",
    aub_reference_signal=vent_timeseries[vent_timeseries.p_vent_idx]["baseline"],
    include_aub=True,
)

In [ ]:
# vent_timeseries.find_ventilator_peaks()

for idx in [vent_timeseries.v_vent_idx, vent_timeseries.f_idx]:
    vent_timeseries.baseline(channel_idxs=[idx], signal_io=("raw", "baseline"))
    vent_timeseries.find_ventilator_peaks(
        channel_io=(idx, idx), peak_set_name="ventilator_breaths", overwrite=True
    )
    vent_timeseries[idx].link_peak_set(
        peak_set_name="ventilator_breaths",
        t_reference_peaks=vent_timeseries[vent_timeseries.p_vent_idx]
        .peaks["Pocc"]
        .peak_df["peak_idx"]
        .to_numpy()
        / vent_timeseries.param["fs"],
        linked_peak_set_name="Pocc",
    )
    vent_timeseries[idx].peaks["Pocc"].detect_on_offset(
        baseline=vent_timeseries[idx]["baseline"],
    )

    vent_timeseries[idx].calculate_time_products(
        peak_set_name="Pocc",
        parameter_name="PTPbreaths",
        aub_reference_signal=vent_timeseries[vent_timeseries.p_vent_idx]["baseline"],
        include_aub=True,
    )
    print(vent_timeseries.labels[idx], vent_timeseries[idx].peaks["Pocc"].peak_df)

## Pocc quality criteria

In [ ]:
for idx in [0, 1, 2]:
    print(vent_timeseries.labels[idx], vent_timeseries[idx].peaks["Pocc"])

# 4. Processing of sEMG signals

## QRS complex detection

In [ ]:
# filter the data
emg_timeseries.filter()

# get the indexes of the ecg and emg channels
emg_timeseries.ecg_idx = next(
    i for i, s in enumerate(emg_timeseries.labels) if "ECG" in s
)
emg_idx = [
    i for i in range(len(emg_timeseries.channels)) if i != emg_timeseries.ecg_idx
]

# Filter the ECG signal and detect the QRS complexes
emg_timeseries.get_ecg_peaks(name="ecg")

# Perform gating to remove the ECG artifacts from the EMG signals
emg_timeseries.gating(
    channel_idxs=emg_idx,
    ecg_peakset_name="ecg",
    gate_width_samples=int(0.1 * emg_timeseries.param["fs"]),
)

# compute the sEA envelope
emg_timeseries.envelope(
    channel_idxs=emg_idx,
    env_window=int(0.2 * emg_timeseries.param["fs"]),
    env_type="rms",
)

## Improved moving slopesum baseline

In [ ]:
# calculate the slopesum moving baseline
emg_timeseries.baseline(
    channel_idxs=emg_idx,
    base_method="slopesum_baseline",
    window_s=int(7.5 * emg_timeseries.param["fs"]),
    ma_window=int(0.5 * emg_timeseries.param["fs"]),
)


## Peak detection

In [ ]:
# detect the sEA peaks
emg_timeseries.detect_emg_breaths(
    channel_idxs=emg_idx,
    # min_peak_width_s=int(0.5 * emg_timeseries.param["fs"]),
    peak_set_name="emg_breaths",
)
# get the on and offsets as baseline crossing points
for idx in emg_idx:
    emg_timeseries[idx].peaks["emg_breaths"].detect_on_offset(
        baseline=emg_timeseries[idx]["baseline"],
        fs=emg_timeseries.fs,
        # method="slope_extrapolation",
    )
    emg_timeseries[idx].link_peak_set(
        peak_set_name="emg_breaths",
        t_reference_peaks=vent_timeseries[vent_timeseries.p_vent_idx]
        .peaks["Pocc"]
        .peak_df["peak_idx"]
        .to_numpy()
        / vent_timeseries.param["fs"],
        linked_peak_set_name="Pocc",
    )

In [ ]:
# link EMG peaks to the closest Pocc peaks
emg_timeseries.link_peak_set(
    channel_idxs=emg_idx,
    peak_set_name="emg_breaths",
    t_reference_peaks=np.asarray(
        vent_timeseries[vent_timeseries.p_vent_idx].peaks["Pocc"].peak_df["peak_idx"]
    )
    / vent_timeseries.param["fs"],
    linked_peak_set_name="Pocc",
)

In [ ]:
# plot the detected Pocc peaks and the linked EMG peaks

n_peaks = len(
    emg_timeseries[emg_idx[0]].peaks["emg_breaths"].peak_df["start_idx"].to_numpy()
)
# plt.close("all")
nrows = 3
ncols = 2
axes = plt.subplots(
    nrows=nrows, ncols=ncols, sharex=True, figsize=(ncols * 6, nrows * 2)
)[1]

axes_emg = axes[:, 0]
axes_vent = axes[:, 1]

emg_timeseries.plot_full(
    axes=axes_emg,
    baseline_bool=True,
)


vent_timeseries.plot_full(
    axes=axes_vent,
    baseline_bool=True,
)

emg_timeseries.plot_markers(
    peak_set_name=["Pocc", "Pocc", "Pocc"], axes=np.transpose([axes_emg])
)
vent_timeseries.plot_markers(
    peak_set_name=["Pocc", "Pocc", "Pocc"], axes=np.transpose([axes_vent]), colors="r"
)

axes_emg[0].set_title("EMG data")
axes_emg[-1].set_xlabel("t (s)")
axes_vent[0].set_title("Ventilator data")
axes_vent[-1].set_xlabel("t (s)")
plt.show()

In [ ]:
plt.close("all")
valid_peaks = (
    vent_timeseries[vent_timeseries.p_vent_idx]
    .peaks["Pocc"]
    .peak_df["valid"]
    .to_numpy()
)
n_peaks = len(valid_peaks)
# n_peaks = len(vent_timeseries[vent_timeseries.p_vent_idx].peaks["Pocc"].peak_df["start_idx"].to_numpy())
vent_idx = [vent_timeseries.p_vent_idx, vent_timeseries.v_vent_idx]
n_peaks_to_plot = min(10, n_peaks)
if n_peaks:
    fig, axes = plt.subplots(
        nrows=4,
        ncols=n_peaks_to_plot,
        sharey="row",
        sharex="col",
        figsize=(6 * n_peaks_to_plot, 6),
        layout="constrained",
    )
    axes_emg = axes[:2, :] if n_peaks_to_plot > 1 else axes[:2]
    axes_vent = axes[2:, :] if n_peaks_to_plot > 1 else axes[2:]
    colors = ["tab:cyan", "tab:orange", "tab:red"]
    margin_s = 5  # margin in seconds around the peak to plot

    emg_timeseries.plot_peaks(
        channel_idxs=emg_idx,
        peak_set_name=["Pocc"],
        axes=axes_emg,
        colors=colors,
        valid_only=True,
        margin_s=margin_s * emg_timeseries.param["fs"],
        n_peaks_to_plot=n_peaks_to_plot,
    )

    emg_timeseries.plot_markers(
        channel_idxs=emg_idx,
        peak_set_name=["Pocc"],
        axes=axes_emg,
        valid_only=True,
        colors=colors,
        n_peaks_to_plot=n_peaks_to_plot,
    )

    for i in range(len(emg_idx)):
        axes_emg[i, 0].set_ylabel(emg_timeseries.labels[emg_idx[i]])

    vent_timeseries.plot_peaks(
        channel_idxs=vent_idx,
        peak_set_name=["Pocc"],
        axes=axes_vent,
        valid_only=True,
        colors=colors,
        margin_s=margin_s * vent_timeseries.param["fs"],
        n_peaks_to_plot=n_peaks_to_plot,
    )

    vent_timeseries.plot_markers(
        channel_idxs=vent_idx,
        peak_set_name=["Pocc"],
        axes=axes_vent,
        valid_only=True,
        colors=colors,
        n_peaks_to_plot=n_peaks_to_plot,
    )

    for i in range(len(vent_idx)):
        axes_vent[i, 0].set_ylabel(vent_timeseries.labels[vent_idx[i]])

    for i in range(n_peaks_to_plot):
        t_peak = (
            vent_timeseries[vent_timeseries.p_vent_idx]
            .peaks["Pocc"]
            .peak_df["peak_idx"]
            .to_numpy()[i]
            / vent_timeseries.param["fs"]
        )
        axes_emg[0, i].set_title(f"Peak {i + 1}\nt = {t_peak:.2f} s")

fig.canvas.header_visible = False  # drop the redundant title bar
fig.canvas.toolbar_visible = True  # pan/zoom/home toolbar
# fig


## ETP calculation

In [ ]:
emg_timeseries.calculate_time_products(
    channel_idxs=emg_idx,
    peak_set_name="Pocc",
    include_aub=True,
    aub_window_s=5 * emg_timeseries.param["fs"],
    parameter_name="ETP",
)

emg_timeseries.calculate_time_products(
    channel_idxs=emg_idx,
    peak_set_name="emg_breaths",
    include_aub=True,
    aub_window_s=5 * emg_timeseries.param["fs"],
    parameter_name="ETP",
)

In [ ]:
vent_timeseries[vent_timeseries.p_vent_idx].peaks["Pocc"].peak_df

## Quality criteria

In [ ]:
for idx in emg_idx:
    # print(f"EMG channel: {emg_timeseries.labels[idx]}")
    emg_timeseries[idx].test_emg_quality(
        peak_set_name="Pocc", parameter_names={"time_product": "ETP"}, verbose=False
    )

    # emg_timeseries[idx].test_emg_quality(peak_set_name="emg_breaths",
    #                                     parameter_names={'time_product': 'ETP'},
    #                                     verbose=False)

vent_timeseries[vent_timeseries.p_vent_idx].test_pocc_quality(
    peak_set_name="Pocc", parameter_names={"time_product": "PTPocc"}, verbose=True
)

In [ ]:
peaks_pocc = (
    vent_timeseries[vent_timeseries.p_vent_idx].peaks["Pocc"].peak_df.copy(deep=True)
)
peaks_pocc["duration_samples"] = peaks_pocc["end_idx"] - peaks_pocc["start_idx"]
peaks_pocc["peak_to_end_samples"] = peaks_pocc["end_idx"] - peaks_pocc["peak_idx"]

In [ ]:
peaks_pocc

# 5. Compute the NMC

In [ ]:
display("p_vent", vent_timeseries[vent_timeseries.p_vent_idx].peaks["Pocc"].peak_df)
display(
    emg_timeseries.labels[emg_idx[0]], emg_timeseries[emg_idx[0]].peaks["Pocc"].peak_df
)
display(
    emg_timeseries.labels[emg_idx[1]], emg_timeseries[emg_idx[1]].peaks["Pocc"].peak_df
)


In [ ]:
vent_peaks = vent_timeseries[vent_timeseries.p_vent_idx].peaks["Pocc"].peak_df
PTPocc = np.abs(vent_peaks["PTPocc"].to_numpy())
valid_vent = vent_peaks["valid"].to_numpy()

NMC = pd.DataFrame(index=vent_peaks.index)
for idx in emg_idx:
    label = emg_timeseries.labels[idx]
    emg_peaks = emg_timeseries[idx].peaks["Pocc"].peak_df
    ETPocc = emg_peaks["ETP"].to_numpy()
    valid_emg = emg_peaks["valid"].to_numpy()

    NMC[(label, "NMC")] = PTPocc / ETPocc
    NMC[(label, "Valid")] = np.logical_and(valid_emg, valid_vent)

NMC.columns = pd.MultiIndex.from_tuples(NMC.columns)
NMC